# Round 4 Manual — "Vanilla Just Isn't Exotic Enough"
## Options on AETHER_CRYSTAL: Pricing, Mispricing Analysis, Trade Recommendation

**Underlying:** AETHER_CRYSTAL (S₀ = 50)  
**Model:** GBM, zero risk-neutral drift, σ = 251% annualized  
**Discretisation:** 4 steps/day × 252 trading days/year = 1008 steps/year  
**2 weeks** = 10 trading days = 40 steps  
**3 weeks** = 15 trading days = 60 steps  

Available products:
- Vanilla calls & puts (2wk and 3wk, various strikes)
- **Chooser option** (3wk expiry, choice at 2wk)
- **Binary put** (pays 10 if S_T < 40 at expiry)
- **Knock-out put** (K=45, barrier=35, knocked out if S ever < 35 at any discrete step)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# ── Problem parameters ───────────────────────────────────────────────────────
S0    = 50.0
sigma = 2.51      # 251% annualized vol
r     = 0.0       # zero risk-neutral drift
CONTRACT_SIZE = 3000

TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY         = 4
STEPS_PER_YEAR        = TRADING_DAYS_PER_YEAR * STEPS_PER_DAY  # 1008

# Exactly as given in the problem
def weeks_to_years(w): return (w * 5) / TRADING_DAYS_PER_YEAR
def steps_for_weeks(w): return int(round(w * 5 * STEPS_PER_DAY))

T2 = steps_for_weeks(2)   # 40  — 2-week options expiry / chooser choice date
T3 = steps_for_weeks(3)   # 60  — 3-week options expiry / chooser expiry
dt = 1.0 / STEPS_PER_YEAR

drift = (r - 0.5 * sigma**2) * dt
diff  = sigma * np.sqrt(dt)

print(f"T2 = {T2} steps  |  T3 = {T3} steps  |  dt = {dt:.8f} years")
print(f"drift/step = {drift:.8f}  |  diffusion/step = {diff:.8f}")
print(f"E[log(S_T3/S0)] = {drift*T3:.5f}  |  Std[log(S_T3/S0)] = {diff*T3**0.5:.5f}")

## 1. Monte Carlo Simulation

In [ ]:
np.random.seed(42)
N = 2_000_000

# Simulate N paths of length T3 steps
# S[:, i] = price at end of step i+1  (S[:,0] is after step 1, S[:,59] is at T3)
Z    = np.random.randn(N, T3)
S    = np.exp(np.log(S0) + np.cumsum(drift + diff * Z, axis=1))  # (N, 60)

S2   = S[:, T2 - 1]        # price at step 40 (2-week mark)
S3   = S[:, T3 - 1]        # price at step 60 (3-week expiry)
Smin = np.min(S, axis=1)   # discrete minimum over all 60 steps

print(f"Paths: {N:,}")
print(f"S2 — mean={S2.mean():.4f}  median={np.median(S2):.4f}  std={S2.std():.4f}")
print(f"S3 — mean={S3.mean():.4f}  median={np.median(S3):.4f}  std={S3.std():.4f}")
print(f"Smin — mean={Smin.mean():.4f}  median={np.median(Smin):.4f}")

## 2. Price Every Product

In [ ]:
def mc(x):
    """Return (mean, 2*SE) of array x."""
    return np.mean(x), 2 * np.std(x) / np.sqrt(len(x))

# Black-Scholes closed form (for cross-check)
def bs_call(S, K, T_yr, v, r=0):
    d1 = (np.log(S/K) + (r + 0.5*v**2)*T_yr) / (v*T_yr**0.5)
    d2 = d1 - v*T_yr**0.5
    return S*stats.norm.cdf(d1) - K*np.exp(-r*T_yr)*stats.norm.cdf(d2)

def bs_put(S, K, T_yr, v, r=0):
    return bs_call(S, K, T_yr, v, r) - S*np.exp(-r*T_yr) + K*np.exp(-r*T_yr)

T2y = weeks_to_years(2)  # 10/252
T3y = weeks_to_years(3)  # 15/252

# ── Payoff arrays ─────────────────────────────────────────────────────────────

# 3-week vanillas
pay_p50_3 = np.maximum(50 - S3, 0)
pay_c50_3 = np.maximum(S3 - 50, 0)
pay_p35_3 = np.maximum(35 - S3, 0)
pay_p40_3 = np.maximum(40 - S3, 0)
pay_p45_3 = np.maximum(45 - S3, 0)
pay_c60_3 = np.maximum(S3 - 60, 0)

# 2-week vanillas
pay_p50_2 = np.maximum(50 - S2, 0)
pay_c50_2 = np.maximum(S2 - 50, 0)

# Chooser: at T2 pick ITM side; payoff realised at T3
# S2 >= 50 → call ITM → becomes call;  S2 < 50 → put ITM → becomes put
pay_cho   = np.where(S2 >= 50, np.maximum(S3 - 50, 0), np.maximum(50 - S3, 0))

# Binary put: pays 10 if S3 < 40
pay_bp    = np.where(S3 < 40, 10.0, 0.0)

# KO put: K=45, barrier=35, discrete monitoring — knocked out if Smin < 35
pay_ko    = np.where(Smin < 35, 0.0, np.maximum(45 - S3, 0))

# ── Fair values ──────────────────────────────────────────────────────────────
products = {
    "AC_50_P":   (pay_p50_3, bs_put( S0, 50, T3y, sigma),  12.00, 12.05),
    "AC_50_C":   (pay_c50_3, bs_call(S0, 50, T3y, sigma),  12.00, 12.05),
    "AC_35_P":   (pay_p35_3, bs_put( S0, 35, T3y, sigma),   4.33,  4.35),
    "AC_40_P":   (pay_p40_3, bs_put( S0, 40, T3y, sigma),   6.50,  6.55),
    "AC_45_P":   (pay_p45_3, bs_put( S0, 45, T3y, sigma),   9.05,  9.10),
    "AC_60_C":   (pay_c60_3, bs_call(S0, 60, T3y, sigma),   8.80,  8.85),
    "AC_50_P_2": (pay_p50_2, bs_put( S0, 50, T2y, sigma),   9.70,  9.75),
    "AC_50_C_2": (pay_c50_2, bs_call(S0, 50, T2y, sigma),   9.70,  9.75),
    "AC_50_CO":  (pay_cho,   bs_call(S0, 50, T3y, sigma) + bs_put(S0, 50, T2y, sigma), 22.20, 22.30),
    "AC_40_BP":  (pay_bp,    None,                           5.00,  5.10),
    "AC_45_KO":  (pay_ko,    None,                           0.15,  0.175),
}

print(f"{'Product':<12} {'MC Fair':>8} {'±2SE':>7} {'BS Fair':>8} {'Bid':>7} {'Ask':>7} {'Eb(buy)':>9} {'Es(sell)':>9}")
print("─" * 78)
fairs = {}
for name, (pay, bs, bid, ask) in products.items():
    m, e2 = mc(pay)
    fairs[name] = m
    bs_str = f"{bs:8.4f}" if bs is not None else "      --"
    print(f"{name:<12} {m:8.4f} {e2:7.4f} {bs_str} {bid:7.3f} {ask:7.4f} {m-ask:9.4f} {bid-m:9.4f}")

## 3. Rubinstein Chooser Decomposition

**Theorem (r=0):** Chooser(K, T₁, T₂) = Call(K, T₂) + Put(K, T₁)

**Proof sketch:**  
At T₁: chooser = max(C, P) = C + max(0, P−C) = C + max(0, K−S_{T₁}) [by put-call parity]  
Taking expectations: E₀[max(C,P)] = E₀[C(S_{T₁}, K, T₂−T₁)] + E₀[max(0, K−S_{T₁})]  
                                   = Call(S₀, K, T₂) + Put(S₀, K, T₁)  ← tower property + put pricing

**Also:** "ITM choice" ≡ "optimal choice" because C > P iff S > K (from put-call parity).

In [ ]:
# Verify Rubinstein
rub = fairs["AC_50_C"] + fairs["AC_50_P_2"]   # Call(T3) + Put(T2)
print(f"Rubinstein:  Call(T3) + Put(T2) = {fairs['AC_50_C']:.5f} + {fairs['AC_50_P_2']:.5f} = {rub:.5f}")
print(f"MC chooser:  {fairs['AC_50_CO']:.5f}")
print(f"Difference:  {fairs['AC_50_CO'] - rub:.6f}  (discrete GBM rounding error, negligible)")
print()
print(f"Market bid:  22.200")
print(f"Edge to SELL at bid: {22.20 - fairs['AC_50_CO']:+.4f} per unit")

# Put-call parity check
print(f"\nPut-call parity (r=0, should be 0):")
print(f"  C(T3) - P(T3) = {fairs['AC_50_C'] - fairs['AC_50_P']:.5f}")
print(f"  C(T2) - P(T2) = {fairs['AC_50_C_2'] - fairs['AC_50_P_2']:.5f}")

# KO details
p_ko = np.mean(Smin < 35)
print(f"\nKO Put details:")
print(f"  P(barrier breached, discrete 60 steps) = {p_ko:.4f} ({p_ko*100:.1f}%)")
print(f"  Standard put(K=45) fair value:           {fairs['AC_45_P']:.4f}")
print(f"  KO put fair value:                       {fairs['AC_45_KO']:.4f}")
print(f"  Discount from knockout:                  {fairs['AC_45_P'] - fairs['AC_45_KO']:.4f}")
print(f"  Edge to BUY at ask 0.175: {fairs['AC_45_KO'] - 0.175:+.5f} per unit")

# Binary put
p_below_40 = np.mean(S3 < 40)
print(f"\nBinary Put details:")
print(f"  P(S_T3 < 40) = {p_below_40:.4f} ({p_below_40*100:.1f}%)")
print(f"  Fair value = 10 × {p_below_40:.4f} = {fairs['AC_40_BP']:.4f}")
print(f"  Edge to SELL at bid 5.00: {5.00 - fairs['AC_40_BP']:+.4f} per unit")

## 4. Visualisation — Fair Values vs Market

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: fair value vs bid/ask ───────────────────────────────────────────────
ax = axes[0]
names = list(products.keys())
mc_vals = [fairs[n] for n in names]
bids    = [products[n][2] for n in names]
asks    = [products[n][3] for n in names]

x = np.arange(len(names))
width = 0.25

bars_mc  = ax.bar(x - width, mc_vals, width, label='MC Fair Value', color='steelblue', alpha=0.85)
bars_bid = ax.bar(x,         bids,    width, label='Market Bid',    color='green',     alpha=0.7)
bars_ask = ax.bar(x + width, asks,    width, label='Market Ask',    color='tomato',    alpha=0.7)

# Highlight mispricings
mispriced = {"AC_50_P_2": "BUY", "AC_50_C_2": "BUY", "AC_50_CO": "SELL",
             "AC_40_BP": "SELL", "AC_45_KO": "BUY"}
for i, name in enumerate(names):
    if name in mispriced:
        ax.axvspan(i - 0.45, i + 0.45, alpha=0.08,
                   color='green' if mispriced[name] == 'BUY' else 'red')
        ax.text(i, max(mc_vals[i], asks[i]) + 0.3,
                mispriced[name], ha='center', fontsize=8,
                color='darkgreen' if mispriced[name] == 'BUY' else 'darkred',
                fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Price (XIRECs)')
ax.set_title('MC Fair Value vs Market Bid/Ask', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# ── Right: edge per unit ──────────────────────────────────────────────────────
ax2 = axes[1]

misprice_details = [
    ("SELL\nAC_50_CO",  22.20 - fairs['AC_50_CO'],  50,  'tomato'),
    ("SELL\nAC_40_BP",  5.00  - fairs['AC_40_BP'],  50,  'tomato'),
    ("BUY\nAC_50_P_2",  fairs['AC_50_P_2'] - 9.75,  50,  'steelblue'),
    ("BUY\nAC_50_C_2",  fairs['AC_50_C_2'] - 9.75,  50,  'steelblue'),
    ("BUY\nAC_45_KO",   fairs['AC_45_KO']  - 0.175, 500, 'steelblue'),
]

labels2   = [d[0] for d in misprice_details]
edges     = [d[1] for d in misprice_details]
vols      = [d[3] for d in misprice_details]
totals    = [d[1]*d[2] for d in misprice_details]
colors2   = [d[3] for d in misprice_details]

bars2 = ax2.bar(range(len(labels2)), totals, color=colors2, alpha=0.85, edgecolor='white', linewidth=1.2)
for i, (bar, tot, e) in enumerate(zip(bars2, totals, edges)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{e:+.3f}/unit\n×{misprice_details[i][2]}vol\n={tot:+.1f}',
             ha='center', va='bottom', fontsize=8)

ax2.axhline(0, color='black', linewidth=1)
ax2.set_xticks(range(len(labels2)))
ax2.set_xticklabels(labels2, fontsize=9)
ax2.set_ylabel('Total Edge (pre ×3000 multiplier)')
ax2.set_title('Edge per Trade (at max volume)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('r4_mispricing.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total edge: {sum(totals):.2f}  →  {sum(totals)*3000:,.0f} expected XIRECs")

## 5. Portfolio PnL Analysis

In [ ]:
# Portfolio: Scenario A (no 3wk call hedge — strictly better E and lower Std)
port = (
      50  * (22.20 - pay_cho)              # SELL 50 choosers at bid 22.20
    + 50  * (pay_p50_2 - 9.75)             # BUY  50 2wk puts  at ask 9.75
    + 50  * (pay_c50_2 - 9.75)             # BUY  50 2wk calls at ask 9.75
    + 50  * (5.00 - pay_bp)                # SELL 50 binary puts at bid 5.00
    + 500 * (pay_ko - 0.175)               # BUY 500 KO puts    at ask 0.175
)

E_port  = np.mean(port)
SD_port = np.std(port)
SE_100  = SD_port / np.sqrt(100)           # SE of 100-sim average (the competition score)
p_pos   = stats.norm.cdf(E_port / SE_100)  # P(100-sim average > 0)

print(f"Expected PnL (per path):       {E_port:.4f}")
print(f"Std (per path):                {SD_port:.2f}")
print(f"SE of 100-sim score:           {SE_100:.2f}")
print(f"P(score > 0 in 100-sim avg):   {p_pos*100:.1f}%")
print(f"Expected XIRECs:               {E_port*CONTRACT_SIZE:,.0f}")
print()

# Simulate many 100-sim competition runs to get the score distribution
np.random.seed(777)
N_trials = 100_000
idx = np.random.randint(0, N, size=(N_trials, 100))
sample_scores = port[idx].mean(axis=1)

print("Distribution of competition scores (100-sim average):")
for p in [5, 25, 50, 75, 95]:
    v = np.percentile(sample_scores, p)
    print(f"  {p:>3}th pct:  {v:>8.1f} units  ({v*CONTRACT_SIZE:>12,.0f} XIRECs)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Left: distribution of 100-sim scores ────────────────────────────────────
ax = axes[0]
ax.hist(sample_scores * CONTRACT_SIZE, bins=200, color='steelblue', alpha=0.75, edgecolor='none')
ax.axvline(E_port * CONTRACT_SIZE, color='gold',    linewidth=2.5, label=f'E = {E_port*CONTRACT_SIZE:,.0f}')
ax.axvline(0,                      color='red',     linewidth=1.5, linestyle='--', label='Break-even')
ax.axvline(np.percentile(sample_scores, 5)  * CONTRACT_SIZE, color='tomato', linewidth=1.5, linestyle=':', label='5th pct')
ax.axvline(np.percentile(sample_scores, 95) * CONTRACT_SIZE, color='lime',   linewidth=1.5, linestyle=':', label='95th pct')
ax.set_xlabel('Score (XIRECs)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of 100-Sim Competition Score', fontweight='bold')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.grid(alpha=0.3)

# ── Middle: S3 distribution + binary put zone ────────────────────────────────
ax2 = axes[1]
log_S3 = np.log(S3)
ax2.hist(log_S3, bins=200, color='steelblue', alpha=0.7, density=True, label='log(S_T3)')
ax2.axvline(np.log(40), color='tomato', linewidth=2, linestyle='--', label='K=40 (binary put)')
ax2.axvline(np.log(45), color='orange', linewidth=2, linestyle='--', label='K=45 (KO put strike)')
ax2.axvline(np.log(35), color='red',    linewidth=2, linestyle=':',  label='B=35 (KO barrier)')
ax2.axvline(np.log(50), color='gold',   linewidth=2, linestyle='--', label='K=50 (chooser/ATM)')
pct_below_40 = np.mean(S3 < 40)
ax2.fill_betweenx([0, 0.6], np.log(S3.min()), np.log(40),
                  alpha=0.12, color='red', label=f'P(S<40)={pct_below_40*100:.1f}%')
ax2.set_xlabel('log(S_T3)')
ax2.set_ylabel('Density')
ax2.set_title('S_T3 Log-Normal Distribution', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# ── Right: path minimum distribution (KO put) ────────────────────────────────
ax3 = axes[2]
log_Smin = np.log(np.maximum(Smin, 0.001))
ax3.hist(log_Smin, bins=200, color='purple', alpha=0.7, density=True, label='log(S_min over 60 steps)')
ax3.axvline(np.log(35), color='red',  linewidth=2.5, linestyle='--', label=f'Barrier=35  P(KO)={p_ko*100:.1f}%')
ax3.fill_betweenx([0, 0.45], log_Smin.min(), np.log(35),
                  alpha=0.15, color='red', label='Knocked out')
ax3.set_xlabel('log(S_min)')
ax3.set_ylabel('Density')
ax3.set_title('Path Minimum Distribution\n(KO put barrier monitoring)', fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('r4_portfolio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Why NOT to Hedge the Chooser with the 3wk Call

The Rubinstein identity says Chooser = Call(T3) + Put(T2). Naively, one might short the chooser and buy both components as a hedge. But in the **full portfolio**, this adds variance:
- In **CALL paths** (S_T2 ≥ 50, ~40%): hedge cancels exactly → Std ≈ 0 ✓
- In **PUT paths** (S_T2 < 50, ~60%): combined payoff = S_T3 − 39.85 (forward exposure) → high Std ✗

In [ ]:
# Scenario B: same + buy 50 3wk calls at ask 12.05
port_B = port + 50 * (pay_c50_3 - 12.05)

eA, sA = np.mean(port), np.std(port)
eB, sB = np.mean(port_B), np.std(port_B)

print("Scenario A (no 3wk call hedge)")
print(f"  E = {eA:.4f}   Std = {sA:.2f}   P(pos 100-sim) = {stats.norm.cdf(eA/(sA/10))*100:.1f}%")
print(f"  Expected XIRECs: {eA*3000:,.0f}")
print()
print("Scenario B (+ buy 50 3wk calls as Rubinstein hedge)")
print(f"  E = {eB:.4f}   Std = {sB:.2f}   P(pos 100-sim) = {stats.norm.cdf(eB/(sB/10))*100:.1f}%")
print(f"  Expected XIRECs: {eB*3000:,.0f}")
print()
print(f"Hedge costs: {(eA-eB)*3000:,.0f} XIRECs in expected PnL AND increases Std by {(sB-sA)/sA*100:.1f}%")
print("=> Scenario A is strictly better (higher E, lower Std, higher P(positive))")

# Explain: PUT paths get forward exposure
put_paths  = S2 < 50
call_paths = S2 >= 50
hedged_put  = (22.20 - pay_cho[put_paths])  + (pay_c50_3[put_paths]  - 12.05)
hedged_call = (22.20 - pay_cho[call_paths]) + (pay_c50_3[call_paths] - 12.05)
print(f"\nIn PUT paths  ({put_paths.mean()*100:.1f}%): hedged PnL = S3-39.85  Std={hedged_put.std():.4f}")
print(f"In CALL paths ({call_paths.mean()*100:.1f}%): hedged PnL = 10.15    Std={hedged_call.std():.4f}  <- perfect cancel")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compare per-path distributions
ax = axes[0]
bins = np.linspace(-400, 500, 150)
ax.hist(port,   bins=bins, alpha=0.6, color='steelblue', density=True, label=f'A (no hedge)  E={eA:.1f}')
ax.hist(port_B, bins=bins, alpha=0.6, color='tomato',    density=True, label=f'B (+ 3wk call)  E={eB:.1f}')
ax.axvline(eA, color='steelblue', linewidth=2)
ax.axvline(eB, color='tomato',    linewidth=2)
ax.axvline(0,  color='black',     linewidth=1.5, linestyle='--')
ax.set_xlabel('Per-path PnL (pre ×3000)')
ax.set_ylabel('Density')
ax.set_title('Portfolio PnL: Hedged vs Unhedged Chooser', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(-400, 500)
ax.grid(alpha=0.3)

# Individual trade contributions
ax2 = axes[1]
trade_names   = ['SELL\n50 CO', 'BUY\n50 P2', 'BUY\n50 C2', 'SELL\n50 BP', 'BUY\n500 KO']
trade_payoffs = [
    50  * (22.20 - pay_cho),
    50  * (pay_p50_2 - 9.75),
    50  * (pay_c50_2 - 9.75),
    50  * (5.00 - pay_bp),
    500 * (pay_ko - 0.175),
]
trade_colors  = ['tomato', 'steelblue', 'steelblue', 'tomato', 'steelblue']
trade_edges   = [np.mean(p) for p in trade_payoffs]
trade_stds    = [np.std(p)  for p in trade_payoffs]

x = np.arange(len(trade_names))
bars = ax2.bar(x, trade_edges, color=trade_colors, alpha=0.8, edgecolor='white')
ax2.errorbar(x, trade_edges, yerr=trade_stds, fmt='none', color='black', capsize=5, linewidth=1.5)
for bar, e in zip(bars, trade_edges):
    ax2.text(bar.get_x() + bar.get_width()/2,
             e + (3 if e >= 0 else -8),
             f'{e:+.1f}', ha='center', fontsize=9, fontweight='bold')
ax2.axhline(0, color='black', linewidth=1)
ax2.set_xticks(x)
ax2.set_xticklabels(trade_names, fontsize=9)
ax2.set_ylabel('Expected PnL contribution (pre ×3000)')
ax2.set_title('Individual Trade Contributions\n(error bars = ±1 Std per path)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('r4_hedge_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Summary & Trade Recommendation

In [ ]:
print("=" * 70)
print("  ROUND 4 MANUAL — FINAL TRADE RECOMMENDATION")
print("=" * 70)
print()

final_trades = [
    ("AC_50_CO",  "SELL", 50,  22.200, fairs['AC_50_CO'],  "Chooser overpriced (Rubinstein: ~21.88)"),
    ("AC_50_P_2", "BUY",  50,   9.750, fairs['AC_50_P_2'], "2wk put underpriced (BS: ~9.87)"),
    ("AC_50_C_2", "BUY",  50,   9.750, fairs['AC_50_C_2'], "2wk call underpriced (BS: ~9.87)"),
    ("AC_40_BP",  "SELL", 50,   5.000, fairs['AC_40_BP'],  "Binary put overpriced (P(S<40)=47.7%→fair=4.77)"),
    ("AC_45_KO",  "BUY",  500,  0.175, fairs['AC_45_KO'],  "KO put underpriced (fair=0.207, edge>20σ)"),
]

total_edge_units = 0
print(f"  {'Product':<12} {'Side':<5} {'Vol':>5}  {'Price':>8} {'Fair':>8} {'Edge/u':>8} {'Total':>8}  Reason")
print("  " + "─" * 100)
for name, side, vol, price, fair, reason in final_trades:
    edge = (price - fair) if side == "SELL" else (fair - price)
    total = edge * vol
    total_edge_units += total
    print(f"  {name:<12} {side:<5} {vol:>5}  {price:>8.4f} {fair:>8.4f} {edge:>8.4f} {total:>8.2f}  {reason}")

print("  " + "─" * 100)
print(f"  {'TOTAL':<12} {'':>5} {'':>5}  {'':>8} {'':>8} {'':>8} {total_edge_units:>8.2f}")
print()
print(f"  × contract size 3000 = {total_edge_units*3000:>12,.0f}  expected XIRECs")
print()
print(f"  Portfolio Std (per path):        {SD_port:>8.2f}")
print(f"  SE of 100-sim score:             {SE_100:>8.2f}")
print(f"  P(score > 0, 100-sim average):   {p_pos*100:>7.1f}%")
print()
print("  SKIP all 3-week vanilla puts/calls (all within market spread, no edge)")
print("  DO NOT buy the 3wk call as Rubinstein hedge (INCREASES variance in full portfolio)")
print()
print("=" * 70)